In [2]:
import sys

sys.path.append("../") # go to parent dir

import pandas as pd
import numpy as np
from custom_helpers_py.get_paths import get_out_folder_house, get_out_folder_senate, get_out_folder_analysis
from os.path import join
from os import listdir
import math

In [3]:
COMPRESSED_SENATE_FILE_PATH = join(get_out_folder_senate(), "compressed", "master_senate.csv")
COMPRESSED_HOUSE_FILE_PATH = join(get_out_folder_house(), "compressed", "master_house.csv")

master_df = pd.concat([
    pd.read_csv(COMPRESSED_SENATE_FILE_PATH),
    pd.read_csv(COMPRESSED_HOUSE_FILE_PATH),
    ], axis=0)

raw_size = master_df.size

# Remove those with null dates, amounts, or action_types
master_df = master_df.dropna(subset=[
    "transaction_date",
    "action_type",
    "amount"
])

no_na_size = master_df.size
removed = raw_size - no_na_size

print("removed", removed)

# Fix tickers
def fix_tickers(in_ticker: None | str):
    if not isinstance(in_ticker, str):
        return np.nan 
    if in_ticker == "--":
        return np.nan
    if "--" in in_ticker or "\n" in in_ticker:
        in_ticker = in_ticker.replace("--", "").replace("\n", "")
    
    if "FB" == in_ticker:
        return "META"
    
    FORBIDDEN_SUB_STRING_LIST = [
        "/",
        " ",
        "%",
        "["
    ]
    for sub_str in FORBIDDEN_SUB_STRING_LIST:
        if sub_str in in_ticker:
            return pd.NA

    return in_ticker

master_df["ticker"] = master_df["ticker"].apply(fix_tickers)

# Process dates
def fix_dates(in_date_str:str):
    month, day, year = in_date_str.split("/")
    if year.startswith("3"):
        year = "2" + year[1:]
    
    if year == "2031":
        year = "2021"
        
    if year == "2202":
        year = "2020"
        
    if year == "2220":
        year = "2020"
    
    
    to_return = "/".join([month, day, year])
    return to_return

master_df["transaction_date"] = master_df["transaction_date"].apply(fix_dates)
master_df["transaction_date"] = pd.to_datetime( master_df["transaction_date"], format="%m/%d/%Y")

# Process amounts
def get_lower_bound_amount(in_amount:str):
    to_return = in_amount
    if "-" in in_amount:
        to_return = in_amount.split("-")[0]
    elif "Over" in in_amount:
        to_return = in_amount.replace("Over", "")

    to_return = to_return.replace("$", "").replace(",", "").strip()
    return float(to_return)

def get_upper_bound_amount(in_amount:str):
    to_return = in_amount
    if "-" in in_amount:
        to_return = in_amount.split("-")[1]
    elif "Over" in in_amount:
        to_return = in_amount.replace("Over", "")

    to_return = to_return.replace("$", "").replace(",", "").strip()
    return float(to_return)

def get_average_amount(in_amount):
    lower_bound, upper_bound = get_lower_bound_amount(in_amount), get_upper_bound_amount(in_amount)

    if upper_bound == np.nan:
        return np.nan
    
    return (upper_bound + lower_bound) / 2
    
master_df["lower_bound_amount"] = master_df["amount"].apply(get_lower_bound_amount)
master_df["upper_bound_amount"] = master_df["amount"].apply(get_upper_bound_amount)
master_df["average_amount"] = master_df["amount"].apply(get_average_amount)

# combine names
master_df["politician_name"] = master_df["politician_first_name"]+ " " + master_df["politician_last_name"]
    
master_df.sort_values(inplace=True, by="transaction_date")

master_df_file_path = join(get_out_folder_analysis(), "master_df.csv")
master_df.to_csv(master_df_file_path, encoding="utf-8", index=False)
master_df

removed 14


,transaction_date,politician_first_name,politician_last_name,office,district,asset_name,asset_type,ticker,owner,action_type,action_type_extra,amount,comments,doc_id,lower_bound_amount,upper_bound_amount,average_amount,politician_name
4267,2011-11-29,ROBERT-P,CORKER-JR,S,NaN,CBL & Associates Properties Inc. - CBL,ST,NaN,SELF,P,NaN,"$1,000,001 - $5,000,000",--,ccf1abc0-3446-4745-a17f-9a6ee9ea32ec,1000001.0,5000000.0,3000000.5,ROBERT-P CORKER-JR
239,2012-02-16,THAD,COCHRAN,S,NaN,"C&J Energy Services, Inc. (NYSE)",NaN,CJES,SELF,S,NaN,"$15,001 - $50,000",--,789c0754-cce2-4dd7-8171-4ac29912e00c,15001.0,50000.0,32500.5,THAD COCHRAN
172,2012-02-16,THAD,COCHRAN,S,NaN,"C&J Energy Services, Inc. (NYSE)",NaN,CJES,SELF,S,NaN,"$15,001 - $50,000",--,4ef44b12-1181-41cc-932a-431ec798f52e,15001.0,50000.0,32500.5,THAD COCHRAN
120,2012-02-27,MR-ALAN-S,LOWENTHAL,H,CA47,E] PASO CORPORATION PREFERRED STOCK,NaN,EP$C,SP,S,NaN,"$1,001 - $15,000",FILING STATUS: NEW,20000945,1001.0,15000.0,8000.5,MR-ALAN-S LOWENTHAL
121,2012-03-20,MR-ALAN-S,LOWENTHAL,H,CA47,E] PASO CORPORATION PREFERRED STOCK,NaN,EP$C,SP,S,NaN,"$1,001 - $15,000",FILING STATUS: NEW,20000945,1001.0,15000.0,8000.5,MR-ALAN-S LOWENTHAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43598,2024-02-21,NANCY,PELOSI,H,CA11,"PALO ALTO NETWORKS, INC.",OP,PANW,SP,P,NaN,"$100,001 - $250,000",NaN,20024542,100001.0,250000.0,175000.5,NANCY PELOSI
43844,2024-02-21,KATHY,MANNING,H,NC06,SB DPC FUND LP,HN,NaN,SP,P,NaN,"$100,001 - $250,000",NaN,20024604,100001.0,250000.0,175000.5,KATHY MANNING
17996,2024-02-22,THOMAS-R,CARPER,S,NaN,US TREASURY BILL,Other Securities,NaN,SP,P,NaN,"$15,001 - $50,000",--,d6290ba2-3a9e-4774-bcd6-6bd2d90e4138,15001.0,50000.0,32500.5,THOMAS-R CARPER
43973,2024-02-26,MARK-DR,GREEN,H,TN07,NGL ENERGY PARTNERS LP COMMON UNITS REPRESENTI...,ST,NGL,SELF,S,NaN,"$50,001 - $100,000",NaN,20024572,50001.0,100000.0,75000.5,MARK-DR GREEN


In [4]:
# Get tickers only
unique_ticker_df = master_df["ticker"].dropna().value_counts().reset_index()
unique_ticker_df.columns = ["ticker", "count"] 
unique_ticker_list = unique_ticker_df.to_dict(orient="records")

unique_ticker_df_file_path = join(get_out_folder_analysis(), "unique_ticker_df.json")
unique_ticker_df.to_json(unique_ticker_df_file_path, orient="records", indent=4)
unique_ticker_list

[{'ticker': 'MSFT', 'count': 829},
 {'ticker': 'AAPL', 'count': 806},
 {'ticker': 'AMZN', 'count': 346},
 {'ticker': 'PARTIAL', 'count': 341},
 {'ticker': 'META', 'count': 310},
 {'ticker': 'DIS', 'count': 309},
 {'ticker': 'JNJ', 'count': 281},
 {'ticker': 'T', 'count': 281},
 {'ticker': 'PFE', 'count': 269},
 {'ticker': 'JPM', 'count': 255},
 {'ticker': 'NVDA', 'count': 247},
 {'ticker': 'INTC', 'count': 244},
 {'ticker': 'VZ', 'count': 236},
 {'ticker': 'PG', 'count': 226},
 {'ticker': 'HD', 'count': 221},
 {'ticker': 'UNH', 'count': 219},
 {'ticker': 'GOOG', 'count': 218},
 {'ticker': 'GE', 'count': 217},
 {'ticker': 'V', 'count': 214},
 {'ticker': 'BAC', 'count': 212},
 {'ticker': 'CVX', 'count': 201},
 {'ticker': 'TSLA', 'count': 200},
 {'ticker': 'WFC', 'count': 198},
 {'ticker': 'SBUX', 'count': 195},
 {'ticker': 'MRK', 'count': 192},
 {'ticker': 'PYPL', 'count': 191},
 {'ticker': 'CVS', 'count': 188},
 {'ticker': 'PEP', 'count': 188},
 {'ticker': 'BA', 'count': 183},
 {'ticker

In [5]:
"""
For each transaction get price of ticker at that date
"""
TICKER_LIST_FOLDER_PATH = join(get_out_folder_analysis(), "ticker_prices")
available_ticker_price = set([tkr.removesuffix(".csv").upper() for tkr in listdir(TICKER_LIST_FOLDER_PATH)])

ticker_df = master_df.dropna(subset="ticker")
ticker_df = ticker_df[ticker_df["ticker"].isin(available_ticker_price)]
ticker_df = ticker_df.sort_values(by="ticker").reset_index(drop=True)

price_df: pd.DataFrame = None
last_ticker = None
ticker_list = ticker_df.to_dict(orient="records")
for row in ticker_list:
    ticker:str = row["ticker"]
    if last_ticker != ticker:
        last_ticker = ticker
        price_file_path = join(TICKER_LIST_FOLDER_PATH, ticker.lower() + ".csv")
        price_df = pd.read_csv(price_file_path)
        price_df["Date"] = pd.to_datetime( price_df["Date"], format="%m/%d/%Y")
    
    transaction_date: pd.Timestamp = pd.to_datetime(row["transaction_date"])
    price_row = price_df[price_df["Date"] == transaction_date]
    if price_row.size != 6:
        continue
    price_row = price_row.iloc[0]

    _, close_price, trade_volume, open_price, high_price, low_price = price_row.to_list() 

    def format_price(in_price):
        if not isinstance(in_price, str):
            return np.nan
        
        in_price = in_price.replace("$", "")
        return float(in_price)

    row["open_price"] = format_price(open_price)
    row["close_price"] = format_price(close_price)
    row["low_price"] = format_price(low_price)
    row["high_price"] = format_price(high_price)
    row["trade_volume"] = format_price(trade_volume)

ticker_df = pd.DataFrame(ticker_list)
    
ticker_df

,transaction_date,politician_first_name,politician_last_name,office,district,asset_name,asset_type,ticker,owner,action_type,...,doc_id,lower_bound_amount,upper_bound_amount,average_amount,politician_name,open_price,close_price,low_price,high_price,trade_volume
0,2018-04-13,LAMAR,SMITH,H,TX21,"AGILENT TECHNOLOGIES, INC.",ST,A,SELF,P,...,20009430,1001.0,15000.0,8000.5,LAMAR SMITH,67.500,67.21,66.880,67.6700,NaN
1,2018-05-16,LAMAR,SMITH,H,TX21,"AGILENT TECHNOLOGIES, INC.",ST,A,SELF,P,...,20009707,1001.0,15000.0,8000.5,LAMAR SMITH,62.510,61.94,61.650,62.9883,NaN
2,2014-02-14,ROBERT-P,CORKER-JR,S,NaN,Agilent Technologies Inc. (NYSE),NaN,A,SELF,S,...,9eeb2456-9f74-4f17-8cc8-d9baa6a24023,1001.0,15000.0,8000.5,ROBERT-P CORKER-JR,NaN,NaN,NaN,NaN,NaN
3,2023-10-16,THOMAS-H,TUBERVILLE,S,NaN,Agilent Technologies,ST,A,JT,S,...,25e0814a-7ea9-41b7-96e6-9c59fbc8c0e3,1001.0,15000.0,8000.5,THOMAS-H TUBERVILLE,111.040,111.50,109.640,112.7900,NaN
4,2014-02-14,ROBERT-P,CORKER-JR,S,NaN,Agilent Technologies Inc. (NYSE),NaN,A,SELF,S,...,e3472690-e844-4636-9cec-cfc3ccf7ab35,1001.0,15000.0,8000.5,ROBERT-P CORKER-JR,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44495,2019-03-25,NICHOLAS-VAN,TAYLOR,H,TX03,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SP,S,...,20011315,1001.0,15000.0,8000.5,NICHOLAS-VAN TAYLOR,33.080,32.97,32.940,33.1040,NaN
44496,2023-07-10,DANIEL,GOLDMAN,H,NY10,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SELF,S,...,20023404,1001.0,15000.0,8000.5,DANIEL GOLDMAN,46.215,46.26,46.130,46.4350,NaN
44497,2023-01-13,DANIEL,GOLDMAN,H,NY10,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SELF,S,...,20022407,15001.0,50000.0,32500.5,DANIEL GOLDMAN,47.860,48.11,47.630,48.1400,NaN
44498,2017-07-11,JOHN,RUTHERFORD,H,FL04,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SELF,P,...,20009815,1001.0,15000.0,8000.5,JOHN RUTHERFORD,29.650,29.79,29.530,29.7900,NaN


In [6]:
# Process further
ticker_df = ticker_df.dropna(subset="open_price")
def get_average_ticker_price(in_row):
    open_price, close_price = in_row["open_price"], in_row["close_price"]
    return (open_price + close_price) / 2
    
ticker_df["average_price"] = ticker_df.apply(get_average_ticker_price, axis=1)
ticker_df = ticker_df[ticker_df["average_price"] > 0]

def get_units_transacted(in_row):
    average_amount, average_price = in_row["average_amount"], in_row["average_price"]
    return math.ceil(average_amount / average_price)

ticker_df["units_transacted"] = ticker_df.apply(get_units_transacted, axis=1)

ticker_df

C:\Users\fdirham\AppData\Local\Temp\ipykernel_48908\3180572723.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_df["average_price"] = ticker_df.apply(get_average_ticker_price, axis=1)


,transaction_date,politician_first_name,politician_last_name,office,district,asset_name,asset_type,ticker,owner,action_type,...,upper_bound_amount,average_amount,politician_name,open_price,close_price,low_price,high_price,trade_volume,average_price,units_transacted
0,2018-04-13,LAMAR,SMITH,H,TX21,"AGILENT TECHNOLOGIES, INC.",ST,A,SELF,P,...,15000.0,8000.5,LAMAR SMITH,67.500,67.21,66.880,67.6700,NaN,67.3550,119
1,2018-05-16,LAMAR,SMITH,H,TX21,"AGILENT TECHNOLOGIES, INC.",ST,A,SELF,P,...,15000.0,8000.5,LAMAR SMITH,62.510,61.94,61.650,62.9883,NaN,62.2250,129
3,2023-10-16,THOMAS-H,TUBERVILLE,S,NaN,Agilent Technologies,ST,A,JT,S,...,15000.0,8000.5,THOMAS-H TUBERVILLE,111.040,111.50,109.640,112.7900,NaN,111.2700,72
5,2019-06-24,DONNA,SHALALA,H,FL27,HYATT HOTELS CORPORATION CLASS A ML,ST,A,SELF,S,...,15000.0,8000.5,DONNA SHALALA,73.330,73.14,72.260,73.6800,NaN,73.2350,110
7,2019-04-03,DONNA,SHALALA,H,FL27,HYATT HOTELS CORPORATION CLASS A,ST,A,SELF,P,...,15000.0,8000.5,DONNA SHALALA,81.540,81.94,81.460,82.0200,NaN,81.7400,98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44495,2019-03-25,NICHOLAS-VAN,TAYLOR,H,TX03,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SP,S,...,15000.0,8000.5,NICHOLAS-VAN TAYLOR,33.080,32.97,32.940,33.1040,NaN,33.0250,243
44496,2023-07-10,DANIEL,GOLDMAN,H,NY10,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SELF,S,...,15000.0,8000.5,DANIEL GOLDMAN,46.215,46.26,46.130,46.4350,NaN,46.2375,174
44497,2023-01-13,DANIEL,GOLDMAN,H,NY10,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SELF,S,...,50000.0,32500.5,DANIEL GOLDMAN,47.860,48.11,47.630,48.1400,NaN,47.9850,678
44498,2017-07-11,JOHN,RUTHERFORD,H,FL04,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SELF,P,...,15000.0,8000.5,JOHN RUTHERFORD,29.650,29.79,29.530,29.7900,NaN,29.7200,270


In [7]:
# Save ticker df
ticker_df_save_path = join(get_out_folder_analysis(), "transaction_with_prices_df.csv")
ticker_df.to_csv(ticker_df_save_path, encoding="utf-8", index=False)
ticker_df

,transaction_date,politician_first_name,politician_last_name,office,district,asset_name,asset_type,ticker,owner,action_type,...,upper_bound_amount,average_amount,politician_name,open_price,close_price,low_price,high_price,trade_volume,average_price,units_transacted
0,2018-04-13,LAMAR,SMITH,H,TX21,"AGILENT TECHNOLOGIES, INC.",ST,A,SELF,P,...,15000.0,8000.5,LAMAR SMITH,67.500,67.21,66.880,67.6700,NaN,67.3550,119
1,2018-05-16,LAMAR,SMITH,H,TX21,"AGILENT TECHNOLOGIES, INC.",ST,A,SELF,P,...,15000.0,8000.5,LAMAR SMITH,62.510,61.94,61.650,62.9883,NaN,62.2250,129
3,2023-10-16,THOMAS-H,TUBERVILLE,S,NaN,Agilent Technologies,ST,A,JT,S,...,15000.0,8000.5,THOMAS-H TUBERVILLE,111.040,111.50,109.640,112.7900,NaN,111.2700,72
5,2019-06-24,DONNA,SHALALA,H,FL27,HYATT HOTELS CORPORATION CLASS A ML,ST,A,SELF,S,...,15000.0,8000.5,DONNA SHALALA,73.330,73.14,72.260,73.6800,NaN,73.2350,110
7,2019-04-03,DONNA,SHALALA,H,FL27,HYATT HOTELS CORPORATION CLASS A,ST,A,SELF,P,...,15000.0,8000.5,DONNA SHALALA,81.540,81.94,81.460,82.0200,NaN,81.7400,98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44495,2019-03-25,NICHOLAS-VAN,TAYLOR,H,TX03,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SP,S,...,15000.0,8000.5,NICHOLAS-VAN TAYLOR,33.080,32.97,32.940,33.1040,NaN,33.0250,243
44496,2023-07-10,DANIEL,GOLDMAN,H,NY10,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SELF,S,...,15000.0,8000.5,DANIEL GOLDMAN,46.215,46.26,46.130,46.4350,NaN,46.2375,174
44497,2023-01-13,DANIEL,GOLDMAN,H,NY10,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SELF,S,...,50000.0,32500.5,DANIEL GOLDMAN,47.860,48.11,47.630,48.1400,NaN,47.9850,678
44498,2017-07-11,JOHN,RUTHERFORD,H,FL04,ZURICH INSURANCE GROUP LIMITED SPONSORED AMERI...,ST,ZURVY,SELF,P,...,15000.0,8000.5,JOHN RUTHERFORD,29.650,29.79,29.530,29.7900,NaN,29.7200,270


In [8]:
"""ANALYSIS STARTS HERE"""
pass

In [9]:
# How many did each person spend trading stocks?
personal_spending_df = master_df.groupby('politician_name').agg({
    "lower_bound_amount": "sum",
    "upper_bound_amount": "sum",
    "average_amount": "sum",
}).reset_index()
personal_spending_df.sort_values(by="average_amount", ascending=False)

,politician_name,lower_bound_amount,upper_bound_amount,average_amount
360,SUZAN-K DELBENE,1.365226e+08,4.574300e+08,2.969763e+08
161,JOSH GOTTHEIMER,8.439557e+07,3.782460e+08,2.313208e+08
340,SCOTT-H PETERS,1.749167e+08,2.813900e+08,2.281533e+08
315,RICK SCOTT,1.017052e+08,2.842750e+08,1.929901e+08
286,NANCY PELOSI,6.214118e+07,2.009400e+08,1.315406e+08
...,...,...,...,...
243,MR-JAMES-R LANGEVIN,1.001000e+03,1.500000e+04,8.000500e+03
322,ROBERT-P CASEY-JR,1.001000e+03,1.500000e+04,8.000500e+03
248,MR-KEVIN YODER,1.001000e+03,1.500000e+04,8.000500e+03
316,ROB WOODALL,1.001000e+03,1.500000e+04,8.000500e+03


In [10]:
# How many did each person spend trading stocks that we know of
personal_spending_df = ticker_df.groupby('politician_name').agg({
    "lower_bound_amount": "sum",
    "upper_bound_amount": "sum",
    "average_amount": "sum",
}).reset_index()
personal_spending_df.sort_values(by="average_amount", ascending=False)

,politician_name,lower_bound_amount,upper_bound_amount,average_amount
125,JOSH GOTTHEIMER,83756208.00,3.721860e+08,2.279711e+08
268,SUZAN-K DELBENE,59669107.00,2.797100e+08,1.696896e+08
132,KELLY LOEFFLER,46480326.00,1.899750e+08,1.182277e+08
211,NANCY PELOSI,51583137.00,1.734450e+08,1.125141e+08
83,GREG GIANFORTE,41912007.00,1.042050e+08,7.305851e+07
...,...,...,...,...
259,STEPHANIE BICE,1001.00,1.500000e+04,8.000500e+03
239,ROBERT-P CASEY-JR,1001.00,1.500000e+04,8.000500e+03
242,ROGER WILLIAMS,1001.00,1.500000e+04,8.000500e+03
184,MR-JIM MCDERMOTT,1001.00,1.500000e+04,8.000500e+03
